# レッスン10: クラスとオブジェクト指向 — 実務コードの読み方が変わる日 🏛️

**実務のコードベースの大半はクラスで書かれています。** ここを越えると、GitHubにある本物のコードが「読める」ようになり始めます。第2部の山場です。

---
## 10-1. なぜクラスが要るのか

銀行口座を「今までの道具だけ」で表現してみると:

```python
account_name = "Tanaka"
account_balance = 5000

def deposit(balance, amount):
    return balance + amount

account_balance = deposit(account_balance, 1000)
```

口座が1つならこれでもOK。でも口座が1000個あったら?「データ(名前・残高)」と「処理(入金・出金)」がバラバラで、どの関数がどのデータ用なのか管理不能になります。

**クラス = データとそれを扱う処理をひとまとめにした設計図。** 設計図から作った実物を**インスタンス(オブジェクト)**と呼びます。

---
## 10-2. クラスの基本形

In [ ]:
class BankAccount:
    """銀行口座"""

    def __init__(self, name, balance=0):
        """口座を新規作成したときに1回だけ走る初期化処理"""
        self.name = name           # self.変数 = このインスタンスが持つデータ(属性)
        self.balance = balance

    def deposit(self, amount):
        """入金"""
        self.balance = self.balance + amount

    def show(self):
        print(f"{self.name} さんの残高: {self.balance:,} 円")


# 設計図から実物(インスタンス)を作る
tanaka = BankAccount("Tanaka", 5000)
sato   = BankAccount("Sato")          # balanceは省略 → デフォルト0

tanaka.deposit(1000)     # インスタンス.メソッド() で呼ぶ
sato.deposit(300)

tanaka.show()            # それぞれが自分のデータを覚えている!
sato.show()

### 文法の整理

- `class 名前:` … 設計図の定義。クラス名は慣例で**大文字始まりのCamelCase**(`BankAccount`)
- `__init__` … インスタンス生成時に自動で走る特別なメソッド(コンストラクタ)
- **`self`** … 「このインスタンス自身」。クラス内のメソッドの第1引数には必ず書く(呼ぶときは書かない)
- `self.balance` … インスタンスが持つデータ(**属性**)。`tanaka.balance` と `sato.balance` は別物

`self` は最初誰もが戸惑うポイントですが、「`tanaka.deposit(1000)` と呼ぶと、Pythonが `deposit(tanaka, 1000)` に変換している。その受け皿が self」と分かれば正体は単純です。

---
## 10-3. 実は今までずっとオブジェクトを使っていた

`"hello".upper()`、`x_list.append(...)`、`T.copy()` — この「値.メソッド()」という書き方、**全部オブジェクトのメソッド呼び出し**でした。文字列もリストもnumpy配列も、誰かが定義したクラスのインスタンスです。つまりあなたはすでにオブジェクト指向コードの「読み方」は使っていて、今日は「作り方」を学んだのです。

---
## 10-4. 実践: 課題をクラスで設計し直す

熱伝導の壁をクラスにすると、実務スタイルの設計がどんなものか体感できます。

In [ ]:
import numpy as np

class Wall:
    """一次元熱伝導を計算する壁"""

    def __init__(self, L, n, lam, rho, c):
        self.L = L
        self.n = n
        self.dx = L / n
        self.a = lam / (rho * c)                  # 熱拡散率
        self.T = np.zeros(n + 1)                  # 温度分布(初期0℃)
        self.x = np.linspace(0, L, n + 1)

    def set_boundary(self, T_edge):
        """両端の温度を設定"""
        self.T[0] = T_edge
        self.T[-1] = T_edge

    def dt_max(self):
        """安定条件によるdt上限 [s]"""
        return self.dx ** 2 / (2 * self.a)

    def step(self, dt):
        """時間をdtだけ進める"""
        r = self.a * dt / self.dx ** 2
        T_new = self.T.copy()
        T_new[1:-1] = self.T[1:-1] + r * (self.T[:-2] - 2*self.T[1:-1] + self.T[2:])
        self.T = T_new


wall = Wall(L=0.3, n=30, lam=1.6, rho=2200, c=880)
wall.set_boundary(1.0)
print("dt上限:", wall.dt_max(), "秒")

for _ in range(9):        # 3分ぶん(20秒×9回)進める。使わないループ変数は _ と書く慣例
    wall.step(20)

print("3分後の端付近:", wall.T[:4])

レッスン6のコードと計算内容は同じですが、使う側は `wall.step(20)` と読むだけで意図が分かる。**「使う人が読みやすい部品を設計する」**のがオブジェクト指向の目的です。

---
## 10-5. 継承 — 既存クラスを拡張する(概念だけ)

既存クラスの機能を受け継いだ新クラスを作れます。実務コード(特にフレームワーク)で頻出するので、「読める」ようにだけしておきましょう。

In [ ]:
class SavingsAccount(BankAccount):        # ← BankAccountを継承
    """利息つき口座。BankAccountの機能は全部そのまま使える"""

    def add_interest(self, rate):
        self.balance = int(self.balance * (1 + rate))

suzuki = SavingsAccount("Suzuki", 10000)
suzuki.deposit(5000)          # 親クラスのメソッドがそのまま使える
suzuki.add_interest(0.02)     # 自分で追加したメソッド
suzuki.show()

---
## 💼 実務メモ: クラスを見たときの読み方ルーチン

実務コードでクラスに出会ったら、この順で読みます:

1. **クラス名とdocstring** — 何を表す部品か
2. **`__init__`** — どんなデータを持つか(属性一覧)
3. **メソッド名を一覧** — 何ができる部品か(中身はまだ読まない!)
4. 必要なメソッドだけ中身を読む

「全部上から読む」は素人、「構造から読む」がプロです。

---
## ✏️ 練習問題 10-A

`Circle` クラスを作ってください:

- `__init__(self, r)` で半径を受け取り `self.r` に保存
- `area()` メソッドで面積(πr²)を返す
- `circumference()` メソッドで円周(2πr)を返す
- 半径5の円を作って両方表示(`math.pi` を使ってOK)

In [ ]:
# ここにコードを書いてください


---
## ✏️ 練習問題 10-B 【例外処理×クラス】

上の `BankAccount` に出金メソッド `withdraw(self, amount)` を追加してください:

- 残高が足りなければ `raise ValueError("残高不足です")`
- 足りていれば残高を減らす
- 正常な出金と、残高不足(try/exceptで受ける)の両方を動かして確認

レッスン8の「入口で検証してfail fast」をクラスの中で使う練習です。

In [ ]:
# ここにコードを書いてください


---
## ✏️ 練習問題 10-C 【総合】

`Wall` クラスを使って(コピーして手直しでOK)、次を実行するコードを書いてください:

1. コンクリート壁(L=0.3, n=30)を作り、両端1℃に設定
2. `step(20)` を繰り返して30分後まで進める(何回stepすればいい?)
3. matplotlibで `wall.x` vs `wall.T` の分布を描く

これができたら、課題のクラス設計版が完成したのと同じです。

In [ ]:
# ここにコードを書いてください


---
## 🎉 レッスン10はここまで!

**今日覚えたこと:**
1. クラス = データ(属性)+処理(メソッド)の設計図。インスタンスが実物
2. `__init__` と `self` の正体
3. 文字列もリストも配列も全部オブジェクトだった
4. 継承 = 既存クラスの拡張(まず読めればOK)
5. クラスは「構造から読む」

**次回 → レッスン11: 実務の作法**(読みやすいコードの書き方=コードレビューで指摘されないコード)

---
### 💡 答え

<details>
<summary>クリックで表示</summary>

```python
# 10-A
import math

class Circle:
    def __init__(self, r):
        self.r = r

    def area(self):
        return math.pi * self.r ** 2

    def circumference(self):
        return 2 * math.pi * self.r

c = Circle(5)
print(c.area())
print(c.circumference())

# 10-B (追加メソッドのみ)
    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("残高不足です")
        self.balance = self.balance - amount

# 動作確認
acc = BankAccount("Tanaka", 5000)
acc.withdraw(3000)
acc.show()                    # 2,000円
try:
    acc.withdraw(99999)
except ValueError as e:
    print("エラー:", e)

# 10-C
import matplotlib.pyplot as plt

wall = Wall(L=0.3, n=30, lam=1.6, rho=2200, c=880)
wall.set_boundary(1.0)
steps = int(30 * 60 / 20)     # 30分 ÷ 20秒 = 90回
for _ in range(steps):
    wall.step(20)

plt.plot(wall.x * 1000, wall.T, marker="o", markersize=3)
plt.xlabel("x [mm]")
plt.ylabel("T [C]")
plt.title("After 30 min")
plt.grid(True)
plt.show()
```
</details>